# Lezione 11 – Pratica: modelli generativi moderni

## Cosa farai in questa pratica

In questa pratica lavori su due elementi centrali dell'autocodificatore variazionale (VAE, dall'inglese *Variational Autoencoder*):

- Il trucco della riparametrizzazione, il meccanismo che permette di addestrare il VAE con la discesa del gradiente nonostante il campionamento casuale del suo spazio latente.
- L'interpolazione nello spazio latente, la proprietà che trasforma quello spazio in una mappa continua in cui spostarsi da un punto all'altro produce transizioni coerenti tra cifre.

Completa il codice dove trovi un commento `# TODO`. Esegui le celle in ordine: l'addestramento dell'esercizio 1 richiede pochi minuti con GPU su Google Colab, e il modello che ne risulta lo riutilizzerai nell'esercizio 2.

## Prerequisito

Devi aver seguito la Lezione 11 ("Modelli generativi moderni: autocodificatori variazionali") e avere chiaro il ruolo di `mu`, `log_var` e della divergenza di Kullback-Leibler (KL) nell'obiettivo di addestramento del VAE.

**Attiva una GPU prima di iniziare:** nel menu di Colab, vai su Runtime > Cambia tipo di runtime > GPU.

## Esercizio 1 — Completa il trucco della riparametrizzazione

### Contesto

Immagina che il tuo team voglia generare variazioni sintetiche di immagini per ampliare un dataset di addestramento, un caso d'uso del VAE in ambito sanitario e biotecnologico che hai visto nella Lezione 11. Per questo ti serve prima un VAE funzionante, e il passo che lo distingue da un autocodificatore normale è il campionamento dentro lo spazio latente.

Il codificatore non produce un unico punto `z` per ogni immagine: produce i parametri di una distribuzione gaussiana, la media `mu` e il logaritmo della varianza `log_var`. Campionare `z` direttamente da quella distribuzione bloccherebbe la propagazione del gradiente, perché non esiste una derivata ben definita di "un numero scelto a caso" rispetto ai parametri che definiscono quel caso.

La soluzione è il trucco della riparametrizzazione: separi il caso (un rumore `epsilon` che non dipende da nessun parametro addestrabile) dalla trasformazione deterministica e derivabile che lo combina con `mu` e `sigma`:

z = mu + sigma * epsilon, con epsilon ~ N(0, I)

Ricorda che la rete predice `log_var` (il logaritmo della varianza), non direttamente `sigma`, quindi devi ricavare `sigma` prima di applicare la formula.

### Cosa devi completare

Nella classe `VAE`, il codificatore (`encoder_hidden`, `fc_mu`, `fc_logvar`) e il decodificatore (`decoder_hidden`, `decoder_out`) sono già scritti. Completa il metodo `reparametrizar`:

1. Calcola `sigma` a partire da `log_var`
2. Genera il rumore `epsilon` con la stessa forma di `sigma`, campionato da una N(0, 1)
3. Combina `mu`, `sigma` ed `epsilon` secondo la formula di riparametrizzazione

Suggerimento: `torch.exp` converte il logaritmo della varianza in varianza (ricorda il fattore 0.5 per passare alla deviazione standard), e `torch.randn_like` genera rumore con la stessa forma e sullo stesso dispositivo (CPU o GPU) del tensore che gli passi.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# --- Datos: MNIST, imágenes de 28x28 en escala de grises, valores en [0, 1] ---
transform = transforms.ToTensor()
train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

LATENT_DIM = 20  # dimensión del espacio latente z


class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=LATENT_DIM):
        super().__init__()
        # Codificador: de la imagen aplanada a una representación oculta compartida
        self.encoder_hidden = nn.Linear(input_dim, hidden_dim)
        # Dos cabezas separadas: una para la media, otra para el log de la varianza
        # (se predice el logaritmo de la varianza, no la varianza directamente,
        # para que la red pueda producir cualquier número real y aun así obtengamos
        # una varianza siempre positiva al exponenciar)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # Decodificador: del espacio latente de vuelta a los píxeles
        self.decoder_hidden = nn.Linear(latent_dim, hidden_dim)
        self.decoder_out = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = F.relu(self.encoder_hidden(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparametrizar(self, mu, log_var):
        # TODO: implementa el truco de reparametrización z = mu + sigma * epsilon
        # 1. Recupera sigma a partir de log_var (log_var = log(sigma^2))
        sigma = ...
        # 2. Genera epsilon ~ N(0, I) con la misma forma y dispositivo que sigma
        epsilon = ...
        # 3. Combina mu, sigma y epsilon en el punto latente z
        z = ...
        return z

    def decode(self, z):
        h = F.relu(self.decoder_hidden(z))
        # Sigmoid porque el decodificador modela una Bernoulli por píxel (valores en [0, 1])
        return torch.sigmoid(self.decoder_out(h))

    def forward(self, x):
        mu, log_var = self.encode(x.view(x.size(0), -1))
        z = self.reparametrizar(mu, log_var)
        x_reconstruida = self.decode(z)
        return x_reconstruida, mu, log_var

In [ ]:
def elbo_loss(x_reconstruida, x, mu, log_var):
    # Término de reconstrucción: log-verosimilitud de una Bernoulli, sumada sobre píxeles
    reconstruccion = F.binary_cross_entropy(
        x_reconstruida, x.view(x.size(0), -1), reduction="sum"
    )
    # KL entre q(z|x) = N(mu, sigma^2) y el prior p(z) = N(0, I), forma cerrada
    kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return reconstruccion + kl  # minimizar esto equivale a maximizar el ELBO


modelo = VAE().to(device)
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)

EPOCHS = 8
for epoch in range(1, EPOCHS + 1):
    modelo.train()
    perdida_total = 0.0
    for x, _ in train_loader:  # las etiquetas no se usan: el VAE es no supervisado
        x = x.to(device)
        optimizador.zero_grad()
        x_reconstruida, mu, log_var = modelo(x)
        perdida = elbo_loss(x_reconstruida, x, mu, log_var)
        perdida.backward()
        optimizador.step()
        perdida_total += perdida.item()

    print(f"Época {epoch}/{EPOCHS} — pérdida media: {perdida_total / len(train_data):.2f}")

# --- Comprobación rápida: reconstrucciones frente a originales ---
modelo.eval()
with torch.no_grad():
    x_test, _ = next(iter(test_loader))
    x_test = x_test.to(device)
    x_reconstruida, _, _ = modelo(x_test)

n = 8
fig, ejes = plt.subplots(2, n, figsize=(n * 1.2, 3))
for i in range(n):
    ejes[0, i].imshow(x_test[i].cpu().squeeze(), cmap="gray")
    ejes[0, i].axis("off")
    ejes[1, i].imshow(x_reconstruida[i].cpu().view(28, 28), cmap="gray")
    ejes[1, i].axis("off")
plt.suptitle("Fila superior: dígitos originales — fila inferior: reconstrucciones del VAE")
plt.show()

<details>
<summary>💡 Vedi soluzione</summary>

```python
def reparametrizar(self, mu, log_var):
    # 1. sigma si ricava da log_var: log_var = log(sigma^2), quindi
    #    sigma = exp(0.5 * log_var). Usare il logaritmo della varianza invece della
    #    varianza diretta evita di vincolare l'uscita della rete a valori positivi.
    sigma = torch.exp(0.5 * log_var)

    # 2. epsilon è il rumore esterno e fisso, indipendente dai parametri
    #    addestrabili: qui vive il caso, isolato dal resto del calcolo.
    epsilon = torch.randn_like(sigma)

    # 3. La trasformazione z = mu + sigma * epsilon è deterministica e derivabile
    #    rispetto a mu e sigma, quindi il gradiente può attraversarla senza problemi.
    z = mu + sigma * epsilon
    return z
```
</details>

## Esercizio 2 — Interpola tra due cifre nello spazio latente

### Contesto

Hai già un VAE addestrato nell'esercizio 1 (la variabile `modelo`). Ora verifica la proprietà che distingue un VAE da un autocodificatore normale: lo spazio latente è una mappa continua, senza zone vuote, quindi spostarsi da un punto all'altro produce una sequenza di immagini sensate, non rumore.

Costruirai una funzione `interpolar` che:

1. Codifica due immagini di cifre diverse e tiene le loro medie `mu_a` e `mu_b` (usa la media, non un campione, per avere un punto di riferimento stabile di ogni cifra)
2. Genera `n_pasos` punti intermedi tra `mu_a` e `mu_b` tramite interpolazione lineare: `z_t = (1 - t) * mu_a + t * mu_b`, con `t` che percorre valori tra zero e uno
3. Decodifica ogni punto intermedio con `modelo.decode(...)`
4. Mostra la sequenza completa su una riga con matplotlib, dalla cifra `a` alla cifra `b`

### Cosa devi completare

L'impalcatura contiene già la funzione `interpolar` con la codifica, la decodifica e la visualizzazione scritte. Completa soltanto il calcolo degli `n_pasos` punti intermedi tra `mu_a` e `mu_b`.

Suggerimento: `torch.linspace(0, 1, n_pasos)` ti fornisce i valori di `t`; ricordati di spostarli sullo stesso `device` di `mu_a` e `mu_b`, e di dar loro la forma adatta affinché il broadcasting combini correttamente ogni `t` con l'intero vettore latente.

In [ ]:
def obtener_par_de_digitos(digito_a, digito_b, dataset):
    """Devuelve una imagen de test para cada dígito solicitado (0-9)."""
    imagen_a = next(x for x, etiqueta in dataset if etiqueta == digito_a)
    imagen_b = next(x for x, etiqueta in dataset if etiqueta == digito_b)
    return imagen_a, imagen_b


def interpolar(modelo, digito_a, digito_b, n_pasos=10):
    modelo.eval()
    with torch.no_grad():
        imagen_a, imagen_b = obtener_par_de_digitos(digito_a, digito_b, test_data)
        x_a = imagen_a.view(1, -1).to(device)
        x_b = imagen_b.view(1, -1).to(device)

        # Codifica ambas imágenes y usa la media como punto de referencia en el espacio latente
        mu_a, _ = modelo.encode(x_a)
        mu_b, _ = modelo.encode(x_b)

        # TODO: genera n_pasos valores de t entre 0 y 1 (torch.linspace) y calcula
        # los puntos intermedios z_t = (1 - t) * mu_a + t * mu_b para cada valor de t.
        # mu_a y mu_b tienen forma (1, latent_dim); da a los valores de t la forma
        # (n_pasos, 1) para que el broadcasting produzca un tensor (n_pasos, latent_dim).
        pasos_t = ...
        puntos_intermedios = ...

        imagenes_generadas = modelo.decode(puntos_intermedios).cpu()

    fig, ejes = plt.subplots(1, n_pasos, figsize=(n_pasos * 1.2, 1.6))
    for i in range(n_pasos):
        ejes[i].imshow(imagenes_generadas[i].view(28, 28), cmap="gray")
        ejes[i].axis("off")
    plt.suptitle(f"Interpolación en el espacio latente: {digito_a} → {digito_b}")
    plt.show()


interpolar(modelo, digito_a=3, digito_b=8, n_pasos=10)

<details>
<summary>💡 Vedi soluzione</summary>

```python
# TODO risolto dentro interpolar(...):

# n_pasos valori di t, distribuiti uniformemente tra 0 e 1, sullo stesso
# dispositivo di mu_a e mu_b, con forma (n_pasos, 1) per poterli combinare
# tramite broadcasting con vettori latenti di forma (1, latent_dim)
pasos_t = torch.linspace(0, 1, n_pasos, device=device).unsqueeze(1)

# Interpolazione lineare: in t=0 ottieni mu_a, in t=1 ottieni mu_b, e nei
# valori intermedi una combinazione ponderata dei due. Il risultato è
# un tensore di forma (n_pasos, latent_dim), un punto latente per passo.
puntos_intermedios = (1 - pasos_t) * mu_a + pasos_t * mu_b
```
</details>

## Chiusura

Hai completato il trucco della riparametrizzazione e verificato con i tuoi occhi che lo spazio latente di un autocodificatore variazionale è una mappa continua: la transizione tra due cifre è graduale, non un salto brusco. La stessa proprietà rende il VAE utile in biotecnologia, nella diagnostica per immagini o nella creazione di contenuti, in ogni caso in cui serva spostarsi in modo controllato tra due punti dotati di significato.

Come passo successivo, prova a cambiare `LATENT_DIM` a due dimensioni, riaddestra e disegna lo spazio latente completo con `plt.scatter`, colorando ogni punto in base alla sua etichetta. Vedrai se cifre simili — come il quattro e il nove — finiscono vicine tra loro nella mappa.

**Torna al forum della Lezione 11 e condividi la tua interpolazione tra due cifre: quali hai scelto e cosa hai osservato nella transizione.**